In [1]:
import sys
from pathlib import Path

print("Python executable:")
print(sys.executable)

print("\nWorking directory:")
print(Path.cwd())

Python executable:
/var/home/connor/Python/$ Projects/materials-pipeline/.venv/bin/python

Working directory:
/var/home/connor/Python/$ Projects/materials-pipeline/notebooks


In [2]:
import importlib.util

package = importlib.util.find_spec("materials_pipeline")

print(package)

ModuleSpec(name='materials_pipeline', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7fb3aa2aedb0>, origin='/var/home/connor/Python/$ Projects/materials-pipeline/src/materials_pipeline/__init__.py', submodule_search_locations=['/var/home/connor/Python/$ Projects/materials-pipeline/src/materials_pipeline'])


In [3]:
from datetime import UTC, datetime

from materials_pipeline.models import (
    CompositionRecord,
    MaterialRecord,
    ProviderName,
    SourceReference,
)

In [4]:
source = SourceReference(
    provider=ProviderName.MATERIALS_PROJECT,
    source_id="mp-149",
    retrieved_at=datetime.now(UTC),
    source_url="https://materialsproject.org/materials/mp-149",
)

composition = CompositionRecord(
    reduced_formula="Si",
    chemical_system="Si",
    element_amounts={"Si": 1.0},
)

material = MaterialRecord(
    composition=composition,
    sources=[source],
)

material

MaterialRecord(record_id=UUID('7566cdbb-33cd-427b-ade4-8d2f68d85a41'), composition=CompositionRecord(reduced_formula='Si', chemical_system='Si', element_amounts={'Si': 1.0}), structure=None, sources=[SourceReference(provider=<ProviderName.MATERIALS_PROJECT: 'materials_project'>, source_id='mp-149', retrieved_at=datetime.datetime(2026, 8, 28, 13, 35, 12, 98714, tzinfo=datetime.timezone.utc), source_url='https://materialsproject.org/materials/mp-149')])

Serialise and dump the material to produce JSON

In [5]:
material.model_dump(mode="json")
print(material.model_dump_json(indent=2))

{
  "record_id": "7566cdbb-33cd-427b-ade4-8d2f68d85a41",
  "composition": {
    "reduced_formula": "Si",
    "chemical_system": "Si",
    "element_amounts": {
      "Si": 1.0
    }
  },
  "structure": null,
  "sources": [
    {
      "provider": "materials_project",
      "source_id": "mp-149",
      "retrieved_at": "2026-08-28T13:35:12.098714Z",
      "source_url": "https://materialsproject.org/materials/mp-149"
    }
  ]
}


## Test validations

In [6]:
from pydantic import ValidationError

try:
    invalid_source = SourceReference(
        provider="unknown_provider",
        source_id="example-001",
        retrieved_at=datetime.now(UTC),
    )
except ValidationError as exc:
    print(exc)

1 validation error for SourceReference
provider
  Input should be 'materials_project', 'nomad' or 'csd' [type=enum, input_value='unknown_provider', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/enum


This should fils because "unknown_provider" isn't one of the values permitted by `ProviderName`

In [7]:
try:
    invalid_composition = CompositionRecord(
        reduced_formula="Si",
        chemical_system="Si",
        element_amounts={"Si": "not a number"},
    )
except ValidationError as exc:
    print(exc)

1 validation error for CompositionRecord
element_amounts.Si
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='not a number', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/float_parsing


This should fils because the element amount cannot be interpreted as a floating-point value.

## A basic materials adapter

In [8]:
# Mock Materials Project record
mp_record = {
    "material_id": "mp-149",
    "formula_pretty": "Si",
    "chemsys": "Si",
    "structure": None,
}

# Provider field       Internal field
# --------------       --------------
# material_id          sources[].source_id
# formula_pretty       composition.reduced_formula
# chemsys              composition.chemical_system
# structure            structure

In [9]:
from materials_pipeline.providers.materials_project import MaterialsProjectProvider

# Create the adapter
provider = MaterialsProjectProvider()

# Transform the mock input
material = provider.transform(mp_record)

material

MaterialRecord(record_id=UUID('6869a05d-f22e-40b5-8fcc-d9219f826cda'), composition=CompositionRecord(reduced_formula='Si', chemical_system='Si', element_amounts={'Si': 1.0}), structure=None, sources=[SourceReference(provider=<ProviderName.MATERIALS_PROJECT: 'materials_project'>, source_id='mp-149', retrieved_at=datetime.datetime(2026, 8, 28, 13, 35, 12, 239642, tzinfo=datetime.timezone.utc), source_url='https://materialsproject.org/materials/mp-149')])

In [10]:
# Inspect the resulting JSON
print(material.model_dump_json(indent=2))

{
  "record_id": "6869a05d-f22e-40b5-8fcc-d9219f826cda",
  "composition": {
    "reduced_formula": "Si",
    "chemical_system": "Si",
    "element_amounts": {
      "Si": 1.0
    }
  },
  "structure": null,
  "sources": [
    {
      "provider": "materials_project",
      "source_id": "mp-149",
      "retrieved_at": "2026-08-28T13:35:12.239642Z",
      "source_url": "https://materialsproject.org/materials/mp-149"
    }
  ]
}


## Testing normalisation

In [11]:
# A deliberately unreduced formula
unreduced_record = {
    "material_id": "example-001",
    "formula_pretty": "Si2O4",
    "chemsys": "Si-O",
    "structure": None,
}

normalised = provider.transform(unreduced_record)

print(normalised.model_dump_json(indent=2))

{
  "record_id": "e20abf2d-f98c-4503-bcd8-a49605ecd5bd",
  "composition": {
    "reduced_formula": "SiO2",
    "chemical_system": "O-Si",
    "element_amounts": {
      "Si": 1.0,
      "O": 2.0
    }
  },
  "structure": null,
  "sources": [
    {
      "provider": "materials_project",
      "source_id": "example-001",
      "retrieved_at": "2026-08-28T13:35:12.254752Z",
      "source_url": "https://materialsproject.org/materials/example-001"
    }
  ]
}


## Now on a real `pymatgen` structure

In [12]:
from pymatgen.core import Lattice, Structure

# Create the silicon lattice
silicon_structure = Structure(
    lattice=Lattice.cubic(5.43),
    species=["Si", "Si"],
    coords=[
        [0.0, 0.0, 0.0],
        [0.25, 0.25, 0.25],
    ],
)

# Create a new mock record
mp_record_with_structure = {
    "material_id": "mp-149",
    "formula_pretty": "Si",
    "chemsys": "Si",
    "structure": silicon_structure,
}

# Transform it
material_with_structure = provider.transform(mp_record_with_structure)

print(material_with_structure.structure)

# Confirm structure can be reconstructed
reconstructed = Structure.from_dict(material_with_structure.structure.pymatgen_data)

print(reconstructed)

assert reconstructed == silicon_structure

pymatgen_data={'@module': 'pymatgen.core.structure', '@class': 'Structure', 'charge': 0.0, 'lattice': {'matrix': [[5.43, 0.0, 0.0], [0.0, 5.43, 0.0], [0.0, 0.0, 5.43]], 'pbc': (True, True, True), 'a': 5.43, 'b': 5.43, 'c': 5.43, 'alpha': 90.0, 'beta': 90.0, 'gamma': 90.0, 'volume': 160.10300699999996}, 'properties': {}, 'sites': [{'species': [{'element': 'Si', 'occu': 1}], 'abc': [0.0, 0.0, 0.0], 'properties': {}, 'label': 'Si', 'xyz': [0.0, 0.0, 0.0]}, {'species': [{'element': 'Si', 'occu': 1}], 'abc': [0.25, 0.25, 0.25], 'properties': {}, 'label': 'Si', 'xyz': [1.3575, 1.3575, 1.3575]}]}
Full Formula (Si2)
Reduced Formula: Si
abc   :   5.430000   5.430000   5.430000
angles:  90.000000  90.000000  90.000000
pbc   :       True       True       True
Sites (2)
  #  SP       a     b     c
---  ----  ----  ----  ----
  0  Si    0     0     0
  1  Si    0.25  0.25  0.25


In [13]:
from materials_pipeline.models import RawEnvelope

print(RawEnvelope.model_json_schema())

{'$defs': {'ProviderName': {'description': 'Data providers supported by the ingestion pipeline', 'enum': ['materials_project', 'nomad', 'csd'], 'title': 'ProviderName', 'type': 'string'}}, 'description': 'Immutable metadata and payload captured from a provider', 'properties': {'provider': {'$ref': '#/$defs/ProviderName'}, 'source_id': {'title': 'Source Id', 'type': 'string'}, 'retrieved_at': {'format': 'date-time', 'title': 'Retrieved At', 'type': 'string'}, 'adapter_version': {'title': 'Adapter Version', 'type': 'string'}, 'payload_sha256': {'title': 'Payload Sha256', 'type': 'string'}, 'payload': {'additionalProperties': True, 'title': 'Payload', 'type': 'object'}}, 'required': ['provider', 'source_id', 'retrieved_at', 'adapter_version', 'payload_sha256', 'payload'], 'title': 'RawEnvelope', 'type': 'object'}


## Test hashing utilities

In [14]:
from materials_pipeline.ingestion import (
    calculate_sha256,
    canonical_json,
    to_json_compatible,
)

In [15]:
record_a = {
    "material_id": "mp-149",
    "formula_pretty": "Si",
}

record_b = {
    "formula_pretty": "Si",
    "material_id": "mp-149",
}

print(canonical_json(record_a))
print(canonical_json(record_b))

print(calculate_sha256(record_a))
print(calculate_sha256(record_b))

assert calculate_sha256(record_a) == calculate_sha256(record_b)

{"formula_pretty":"Si","material_id":"mp-149"}
{"formula_pretty":"Si","material_id":"mp-149"}
ca93511503b5f1617d6e9e2b430c4c6dca7c5d407f5bb94d0f4be21d4c9c351b
ca93511503b5f1617d6e9e2b430c4c6dca7c5d407f5bb94d0f4be21d4c9c351b


In [16]:
compatible_record = to_json_compatible(mp_record_with_structure)

type(compatible_record["structure"])

dict

In [17]:
type(mp_record_with_structure["structure"])

pymatgen.core.structure.Structure

# Stage 2: End-to-end Ingestion

## Required imports

In [18]:
from pathlib import Path

from materials_pipeline.pipeline import IngestionPipeline
from materials_pipeline.providers.materials_project import (
    MaterialsProjectProvider,
)

## Find the project root

In [19]:
current_directory = Path.cwd()

project_root = (
    current_directory.parent
    if current_directory.name == "notebooks"
    else current_directory
)

print(project_root)
print((project_root / "pyproject.toml").exists())

/var/home/connor/Python/$ Projects/materials-pipeline
True


## Create the pipeline, provider and mock record

In [20]:
pipeline = IngestionPipeline(project_root=project_root)

provider = MaterialsProjectProvider()

mp_record = {
    "material_id": "mp-149",
    "formula_pretty": "Si",
    "chemsys": "Si",
    "structure": None,
}

## Run ingestion

In [21]:
result = pipeline.ingest(
    provider=provider,
    raw=mp_record,
)

result

IngestionResult(status='accepted', provider=<ProviderName.MATERIALS_PROJECT: 'materials_project'>, source_id='mp-149', raw_path='/var/home/connor/Python/$ Projects/materials-pipeline/data/raw/materials_project/mp-149/20260828T133512309590Z.json', normalised_path='/var/home/connor/Python/$ Projects/materials-pipeline/data/normalised/257e33b5-4342-435f-8913-b2f1bca9c4e7.json', material=MaterialRecord(record_id=UUID('257e33b5-4342-435f-8913-b2f1bca9c4e7'), composition=CompositionRecord(reduced_formula='Si', chemical_system='Si', element_amounts={'Si': 1.0}), structure=None, sources=[SourceReference(provider=<ProviderName.MATERIALS_PROJECT: 'materials_project'>, source_id='mp-149', retrieved_at=datetime.datetime(2026, 8, 28, 13, 35, 12, 310082, tzinfo=datetime.timezone.utc), source_url='https://materialsproject.org/materials/mp-149')]), issues=[])

In [22]:
print(result.model_dump_json(indent=2))

{
  "status": "accepted",
  "provider": "materials_project",
  "source_id": "mp-149",
  "raw_path": "/var/home/connor/Python/$ Projects/materials-pipeline/data/raw/materials_project/mp-149/20260828T133512309590Z.json",
  "normalised_path": "/var/home/connor/Python/$ Projects/materials-pipeline/data/normalised/257e33b5-4342-435f-8913-b2f1bca9c4e7.json",
  "material": {
    "record_id": "257e33b5-4342-435f-8913-b2f1bca9c4e7",
    "composition": {
      "reduced_formula": "Si",
      "chemical_system": "Si",
      "element_amounts": {
        "Si": 1.0
      }
    },
    "structure": null,
    "sources": [
      {
        "provider": "materials_project",
        "source_id": "mp-149",
        "retrieved_at": "2026-08-28T13:35:12.310082Z",
        "source_url": "https://materialsproject.org/materials/mp-149"
      }
    ]
  },
  "issues": []
}


## Inspect the generated files

### Locate the files

In [23]:
print(result.raw_path)
print(result.normalised_path)

/var/home/connor/Python/$ Projects/materials-pipeline/data/raw/materials_project/mp-149/20260828T133512309590Z.json
/var/home/connor/Python/$ Projects/materials-pipeline/data/normalised/257e33b5-4342-435f-8913-b2f1bca9c4e7.json


### Read the raw snapshot

In [24]:
raw_json = Path(result.raw_path).read_text(encoding="utf-8")

print(raw_json)

{
  "provider": "materials_project",
  "source_id": "mp-149",
  "retrieved_at": "2026-08-28T13:35:12.309590Z",
  "adapter_version": "0.1.0",
  "payload_sha256": "6d297791162f0586f3f161447919f3d13e0c3d9b47c6e02a10a6b0f3bac0686c",
  "payload": {
    "material_id": "mp-149",
    "formula_pretty": "Si",
    "chemsys": "Si",
    "structure": null
  }
}



### Read the normalised output

In [25]:
normalised_json = Path(result.normalised_path).read_text(encoding="utf-8")

print(normalised_json)

{
  "record_id": "257e33b5-4342-435f-8913-b2f1bca9c4e7",
  "composition": {
    "reduced_formula": "Si",
    "chemical_system": "Si",
    "element_amounts": {
      "Si": 1.0
    }
  },
  "structure": null,
  "sources": [
    {
      "provider": "materials_project",
      "source_id": "mp-149",
      "retrieved_at": "2026-08-28T13:35:12.310082Z",
      "source_url": "https://materialsproject.org/materials/mp-149"
    }
  ]
}



Note that the raw snapshot contains:
* provider metadata
* source ID
* retrieval timestamp
* adapter version
* checksum
* original provider payload


...and the normalised file contains:
* internal UUID
* common composition model
* common structure model
* source references

## Try a rejected model

### Create a record without a Materials Project identifier

In [26]:
invalid_record = {
    "formula_pretty": "Si",
    "chemsys": "Si",
    "structure": None,
}

### Ingest the record

In [27]:
rejected_result = pipeline.ingest(
    provider=provider,
    raw=invalid_record,
)

print(rejected_result.model_dump_json(indent=2))

{
  "status": "rejected",
  "provider": "materials_project",
  "source_id": "unknown-source",
  "raw_path": "/var/home/connor/Python/$ Projects/materials-pipeline/data/raw/materials_project/unknown-source/20260828T133512338655Z.json",
  "normalised_path": null,
  "material": null,
  "issues": [
    {
      "code": "TRANSFORMATION_FAILED",
      "severity": "error",
      "message": "Required field \"material_id\" must be a non-empty string.",
      "field": null
    }
  ]
}


The important result is that the invalid record was still preserved under:
```text
data/raw/materials_project/unknown-source/
```